# Ejemplo: Pipeline AWS para llamadas de soporte

Este notebook muestra, de forma didáctica, cómo se vería un flujo para:

1. **Transcribir llamadas de soporte** con Amazon Transcribe.
2. **Analizar sentimiento del cliente** con Amazon Comprehend.
3. **Buscar soluciones en una base de conocimiento** con Amazon Kendra.

> Nota: el notebook incluye una versión simulada para poder ejecutarse sin AWS. También incluye ejemplos de código con `boto3` para conectarlo a servicios reales.

## 1. Instalación opcional

Si quieres usar servicios reales de AWS, instala `boto3` y configura tus credenciales.

```bash
pip install boto3 pandas
aws configure
```

In [ ]:
import pandas as pd
from pprint import pprint

## 2. Entrada: llamada de soporte

En producción, aquí tendrías un archivo de audio en S3, por ejemplo:

```text
s3://mi-bucket-soporte/llamadas/call_001.wav
```

Para este ejemplo usaremos una transcripción simulada.

In [ ]:
transcript = """
Cliente: Estoy muy molesto porque mi pedido aparece como entregado, pero nunca llegó.
Agente: Lamento mucho el inconveniente. Voy a revisar el estado del pedido.
Cliente: Ya llevo tres días esperando y necesito una solución hoy.
Agente: Entiendo. Podemos abrir una investigación con la transportadora o enviar un reemplazo si aplica.
""".strip()

print(transcript)

## 3. Amazon Transcribe: ejemplo real con boto3

Este código muestra cómo iniciar un trabajo de transcripción real. No se ejecuta automáticamente porque requiere AWS, S3 y permisos configurados.

In [ ]:
# Ejemplo real con AWS Transcribe
# Descomenta y ajusta valores si tienes credenciales AWS configuradas.

# import boto3
# import time

# transcribe = boto3.client('transcribe', region_name='us-east-1')

# job_name = 'support-call-001'
# media_uri = 's3://mi-bucket-soporte/llamadas/call_001.wav'

# transcribe.start_transcription_job(
#     TranscriptionJobName=job_name,
#     Media={'MediaFileUri': media_uri},
#     MediaFormat='wav',
#     LanguageCode='es-US'
# )

# while True:
#     result = transcribe.get_transcription_job(TranscriptionJobName=job_name)
#     status = result['TranscriptionJob']['TranscriptionJobStatus']
#     print('Estado:', status)
#     if status in ['COMPLETED', 'FAILED']:
#         break
#     time.sleep(10)

# result

## 4. Amazon Comprehend: análisis de sentimiento

Primero simularemos un resultado de sentimiento. Luego dejamos el ejemplo real con AWS.

In [ ]:
# Simulación simple de sentimiento para fines educativos
negative_words = ['molesto', 'nunca llegó', 'esperando', 'inconveniente', 'solución hoy']
positive_words = ['gracias', 'excelente', 'perfecto', 'resuelto']

text_lower = transcript.lower()
negative_score = sum(word in text_lower for word in negative_words)
positive_score = sum(word in text_lower for word in positive_words)

if negative_score > positive_score:
    sentiment = 'NEGATIVE'
elif positive_score > negative_score:
    sentiment = 'POSITIVE'
else:
    sentiment = 'NEUTRAL'

sentiment_result = {
    'Sentiment': sentiment,
    'NegativeScore': negative_score,
    'PositiveScore': positive_score
}

sentiment_result

In [ ]:
# Ejemplo real con Amazon Comprehend
# Descomenta si tienes AWS configurado.

# import boto3
# comprehend = boto3.client('comprehend', region_name='us-east-1')

# response = comprehend.detect_sentiment(
#     Text=transcript,
#     LanguageCode='es'
# )

# pprint(response)

## 5. Base de conocimiento simulada

En producción, estos documentos estarían indexados en Amazon Kendra. Aquí simularemos una pequeña base de conocimiento con artículos de soporte.

In [ ]:
knowledge_base = pd.DataFrame([
    {
        'id': 'KB001',
        'title': 'Pedido aparece como entregado pero el cliente no lo recibió',
        'content': 'Verificar tracking, confirmar dirección, abrir investigación con transportadora y ofrecer reemplazo si aplica.'
    },
    {
        'id': 'KB002',
        'title': 'Cliente solicita devolución de dinero',
        'content': 'Validar política de reembolso, estado del pedido y método de pago antes de procesar la devolución.'
    },
    {
        'id': 'KB003',
        'title': 'Demora en envío',
        'content': 'Revisar fecha estimada, transportadora, eventos de tracking y comunicar nueva fecha al cliente.'
    }
])

knowledge_base

## 6. Búsqueda tipo Kendra simulada

Haremos una búsqueda simple por coincidencia de palabras clave. Kendra real hace esto de forma mucho más avanzada usando búsqueda semántica, ranking y conectores a fuentes de datos.

In [ ]:
def simple_search(query, kb):
    query_terms = set(query.lower().replace(',', '').replace('.', '').split())
    results = []

    for _, row in kb.iterrows():
        text = f"{row['title']} {row['content']}".lower()
        score = sum(term in text for term in query_terms)
        results.append({
            'id': row['id'],
            'title': row['title'],
            'content': row['content'],
            'score': score
        })

    return pd.DataFrame(results).sort_values('score', ascending=False)

query = 'pedido entregado pero nunca llegó solución transportadora reemplazo'
search_results = simple_search(query, knowledge_base)
search_results

In [ ]:
# Ejemplo real con Amazon Kendra
# Descomenta si tienes un índice de Kendra configurado.

# import boto3
# kendra = boto3.client('kendra', region_name='us-east-1')

# index_id = 'TU_KENDRA_INDEX_ID'
# response = kendra.query(
#     IndexId=index_id,
#     QueryText=query
# )

# for item in response['ResultItems']:
#     print(item.get('Type'))
#     print(item.get('DocumentTitle', {}).get('Text'))
#     print(item.get('DocumentExcerpt', {}).get('Text'))
#     print('-' * 80)

## 7. Resultado final del pipeline

Ahora juntamos las salidas: transcripción, sentimiento y solución recomendada.

In [ ]:
top_result = search_results.iloc[0].to_dict()

pipeline_output = {
    'transcripcion': transcript,
    'sentimiento_cliente': sentiment_result['Sentiment'],
    'articulo_recomendado': {
        'id': top_result['id'],
        'titulo': top_result['title'],
        'solucion': top_result['content']
    }
}

pprint(pipeline_output)

## 8. Arquitectura recomendada en AWS

Una arquitectura típica sería:

```text
Amazon Connect / archivo de llamada
        ↓
Amazon S3
        ↓
Amazon Transcribe / Transcribe Call Analytics
        ↓
Amazon Comprehend
        ↓
Amazon Kendra
        ↓
Dashboard, CRM o aplicación del agente
```

Servicios adicionales útiles:

- **AWS Lambda**: orquestar pasos del pipeline.
- **Amazon S3**: almacenar audios y transcripciones.
- **Amazon CloudWatch**: monitoreo y logs.
- **Amazon QuickSight**: dashboard de métricas de sentimiento y temas frecuentes.
- **Amazon Connect Contact Lens**: alternativa integrada para contact center.